In [1]:
import sys,os
import numpy as np
from crispy.tools.initLogger import getLogger
log = getLogger('crispy')

This notebook is intended to be a sandbox that demonstrates functionality illustrated at the following webpage:
https://mjrfringes.github.io/crispy/notebooks/Introduction.html

## 1. Initialization

In [2]:
os.chdir("C:/Users/ebray/Github Repos/crispy/crispy/WFIRST/")
from params import Params
par = Params()
par.hdr

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                    8 / array data type                                
NAXIS   =                    0 / number of array dimensions                     
EXTEND  =                    T                                                  
COMMENT                                                                         
COMMENT ************************************************************            
COMMENT ********************** General parameters ******************            
COMMENT ************************************************************            
COMMENT                                                                         
NLENS   =                  108 / # lenslets across array                        
PITCH   =             0.000174 / Lenslet pitch (meters)                         
INTERLAC=                  2.0 / Interlacing                                    
PHILENS =    26.565051177077

In [3]:
dir(par)

['BW',
 'CIC',
 'EMGain',
 'EMStats',
 'FWHM',
 'FWHMlam',
 'Nreads',
 'PCbias',
 'PCmode',
 'PSFLetPositions',
 'PhCountEff',
 'QE',
 'R',
 'RN',
 'Traps',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'codeRoot',
 'convolve',
 'dark',
 'exportDir',
 'filelist',
 'gaussian',
 'gaussian_hires',
 'hdr',
 'interlace',
 'lamlist',
 'lenslet_sampling',
 'lenslet_wav',
 'lensletlam',
 'lensletsampling',
 'lifefraction',
 'losses',
 'makeHeader',
 'nchanperspec_lstsq',
 'nlens',
 'nonoise',
 'npix',
 'npixperdlam',
 'philens',
 'pinhole',
 'pitch',
 'pixsize',
 'poisson',
 'pol',
 'prefix',
 'pxperdetpix',
 'saveDetector',
 'saveLensletPlane',
 'savePoly',

## 2. Create the flatfield

In [4]:
from crispy.unitTests import testCreateFlatfield
help(testCreateFlatfield)

Help on function testCreateFlatfield in module crispy.unitTests:

testCreateFlatfield(par, pixsize=0.1, npix=512, pixval=1.0, Nspec=45, outname='flatfield.fits', useQE=True, method='optext', maxflux=400, bg=10)
    Creates a polychromatic flatfield 
    #"And then pass it through the IFS? I feel like that makes the name of this function misleading.
    # Maybe it should be broken up into two functions titled 'testCreateFlatfieldCube' and 'testProcessFlatfieldCube' " - Evan Bray 2025/08/12
    
    Parameters
    ----------
    par :   Parameter instance
        Contains all IFS parameters
    pixsize:   float
       Pixel scale (lam/D)
    npix: int
        Each input frame has a pixel size npix x npix
    pixval: float
        Each input frame has a unform value pixval in photons per second per nm of bandwidth
    Nspec: float
        Optional input forcing the number of wavelengths bins used
    outname: string
        Name of flatfield image
    useQE: boolean 
        Whether to ta

#### 2.1 Determine wavelengths at which to make the flatfield

In [5]:
from crispy.tools.reduction import calculateWaveList
#help(calculateWaveList)
lam_midpts,lam_endpts = calculateWaveList(par)
print(lam_midpts)

crispy - INFO - Reading in lam_list from ..//ReferenceFiles/wavecalR50_770/lamsol.dat
crispy - INFO - Reduced cube will have 18 wavelength bins
[704.25443039 711.41747416 718.65337398 725.96287087 733.34671341
 740.80565776 748.34046779 755.95191515 763.64077932 771.40784771
 779.25391575 787.17978694 795.18627299 803.27419382 811.44437771
 819.69766139 828.03489005 836.45691752]


#### 2.2 Actually create the flatfield

In [6]:
testCreateFlatfield(par,useQE=False)

crispy - INFO - Reading in lam_list from ..//ReferenceFiles/wavecalR50_770/lamsol.dat
crispy - INFO - Reduced cube will have 44 wavelength bins
crispy - INFO - The number of input pixels per lenslet is 5.016181
crispy - INFO - Using PSFlet gaussian approximation
crispy - WARNING - Assuming endpoints wavelist is given
crispy - INFO - Creating Gaussian PSFLet templates
crispy - INFO - Writing data to ..//SimResults/detectorFramepoly.fits
crispy - INFO - Done.
crispy - INFO - Performance: 41 seconds total
crispy - INFO - Writing data to ..//unitTestsOutputs/flatfield.fits


In [7]:
# And also display the newly-appended-to header parameters
par.hdr

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                    8 / array data type                                
NAXIS   =                    0 / number of array dimensions                     
EXTEND  =                    T                                                  
COMMENT                                                                         
COMMENT ************************************************************            
COMMENT ********************** General parameters ******************            
COMMENT ************************************************************            
COMMENT                                                                         
NLENS   =                  108 / # lenslets across array                        
PITCH   =             0.000174 / Lenslet pitch (meters)                         
INTERLAC=                  2.0 / Interlacing                                    
PHILENS =    26.565051177077

In [8]:
par.lenslet_sampling

0.5

#### 2.3 Display some results

##### 2.3.1 Starting with the input flatfield cube passed through the IFS

In [9]:
from crispy.tools.image import Image
import matplotlib.pyplot as plt
plt.close('all')
%matplotlib qt
fig,ax = plt.subplots(figsize=(8,7))
image_filepath = par.unitTestsOutputs+'/flatfield.fits'
img = Image(image_filepath).data
im = ax.imshow(img,cmap='gray')
cbar = plt.colorbar(im)
fig.tight_layout()
plt.show()

# Also display a zoomed-in portion in higher detail. 
plt.figure(figsize=(6,6))
subsize = 50
plt.imshow(img[par.npix//2-subsize:par.npix//2+subsize,par.npix//2-subsize:par.npix//2+subsize],cmap='gray')
plt.colorbar()
plt.show()

crispy - INFO - Read data from HDU 1 of ..//unitTestsOutputs/flatfield.fits


### 3 Simulate Detector Readout

In [10]:
from crispy.tools.detector import readDetector,averageDetectorReadout
par.nonoise=False  # turn off photon counting otherwise things are strange
par.EMStats =False # turn off EM register statistics
par.PCmode = False # turn off photon counting threshold
read=readDetector(par,Image(image_filepath),inttime=100)


plt.figure(figsize=(6,6))
subsize = 15

plt.imshow(read[par.npix//2-subsize:par.npix//2+subsize,par.npix//2-subsize:par.npix//2+subsize],cmap='gray')
plt.colorbar()

crispy - INFO - Read data from HDU 1 of ..//unitTestsOutputs/flatfield.fits


In [11]:
# Let’s save the noisified frame to a new name.
newImage = Image(data=read,header=par.hdr)
newImage.write(par.unitTestsOutputs+'/flatfield_noise.fits',overwrite=True)

crispy - INFO - Writing data to ..//unitTestsOutputs/flatfield_noise.fits


### 4 Spectral Extraction

In [12]:
# The reduction step is straightforward, as long as the wavelength calibration is good.
from crispy.IFS import reduceIFSMap
cube = reduceIFSMap(par,par.unitTestsOutputs+'/flatfield_noise.fits')

crispy - INFO - Read data from HDU 1 of ..//unitTestsOutputs/flatfield_noise.fits
crispy - INFO - Mean, median, std: (2880.488, 528.0, 4514.8486)
crispy - INFO - Subtracting median from image
crispy - INFO - Reading in lam_list from ..//ReferenceFiles/wavecalR50_770/lamsol.dat
crispy - INFO - Reduced cube will have 18 wavelength bins
crispy - INFO - Elapsed time: 1.325820s


In [16]:
# Now we can display the cube interactively, or look it up with DS9 (it is located in par.exportDir)

import ipywidgets
def plt_ifs_optext(wchan):
    plt.clf()  # Clear the current figure
    im = plt.imshow(cube.data[wchan-1,:,:], cmap='gist_heat')
    plt.colorbar(im)  # Add a new colorbar for the current image
ipywidgets.interact(plt_ifs_optext, wchan=(1,cube.data.shape[0]));

interactive(children=(IntSlider(value=9, description='wchan', max=18, min=1), Output()), _dom_classes=('widget…